In [2]:
%load_ext autoreload
%autoreload 2


import numpy as np
import pandas as pd

from assignment_1.modules.models import (
    fit_gap_model,
    item_first_seen_bound,
    predict_gaps,
)
from assignment_1.modules.util import clip_timestamps, load_csv

DATA_DIR = "../data"  # notebook runs from notebooks/

In [3]:
TASK_TRAIN_FILE = "task_6_train.csv"
TASK_TEST_FILE = "task_6_test.csv"
train = load_csv(TASK_TRAIN_FILE, DATA_DIR)
test = load_csv(TASK_TEST_FILE, DATA_DIR)

In [4]:
train.head()

,user,item,timestamp,rating
0,u4993107552,i0025361823,775725363,3
1,u4993107552,i0053955695,804585652,4
2,u4993107552,i9175305854,807256252,4
3,u4993107552,i5617235264,1122879726,4
4,u4993107552,i7388390860,1151738914,1


In [5]:
test.head()

,user,item,delta
0,u4993107552,i5962907281,NaN
1,u9202480616,i2564745237,NaN
2,u3005263451,i9062564224,NaN
3,u2876968517,i2692895515,NaN
4,u5049051907,i5888634296,NaN


In [6]:
print(f"train {train.shape}, test {test.shape}, test users unique: {test.user.is_unique}")
print(f"test users in train: {test.user.isin(train.user).mean():.1%}")

hist = train.groupby("user")["item"].apply(set)
in_hist = [i in hist.get(u, set()) for u, i in zip(test.user, test.item)]
print(f"test item already in user's train history: {np.mean(in_hist):.1%}")

s = train.sort_values(["user", "timestamp"], kind="stable")
d = s.groupby("user")["timestamp"].diff().dropna()          # consecutive gaps, seconds
print(f"{len(d):,} gaps; negative: {(d < 0).sum()}, zero: {(d == 0).mean():.1%}")
print((d / 86400).describe(percentiles=[.1, .25, .5, .75, .9, .99]).round(1))   # in days

n_hist = train.groupby("user").size()
print(f"users with only 1 train row (no gaps to learn from): {(n_hist == 1).mean():.1%}")

train (1301245, 4), test (55403, 3), test users unique: True
test users in train: 100.0%
test item already in user's train history: 0.0%
1,245,842 gaps; negative: 0, zero: 0.0%
count    1245842.0
mean          70.9
std          878.6
min            0.0
10%            1.0
25%            4.0
50%           13.0
75%           46.0
90%          143.9
99%          829.0
max       720204.0
Name: timestamp, dtype: float64
users with only 1 train row (no gaps to learn from): 0.0%


In [7]:
s = train.sort_values(["user", "timestamp"], kind="stable").copy()
s["gap"] = s.groupby("user")["timestamp"].diff()          # gap before each row (NaN on first row)

is_last = ~s["user"].duplicated(keep="last")
target = s.loc[is_last, ["user", "gap"]].set_index("user")["gap"]   # each user's last gap = what to predict
hist_gaps = s.loc[~is_last].dropna(subset=["gap"])                  # gaps strictly before it

per_user_median = hist_gaps.groupby("user")["gap"].median()
global_median = hist_gaps["gap"].median()
pred = target.index.map(per_user_median).fillna(global_median)      # users with no earlier gap -> global

err = target.to_numpy() - pred.to_numpy()
print(f"users with no earlier gap (fallback): {target.index.map(per_user_median).isna().mean():.1%}")
print(f"MAE  = {np.abs(err).mean():,.0f}")
print(f"MSE  = {(err ** 2).mean():,.0f}")
print("public baseline = 13,563,799")

users with no earlier gap (fallback): 0.0%
MAE  = 12,052,136
MSE  = 797,627,092,673,614
public baseline = 13,563,799


In [8]:
m_global = hist_gaps["gap"].median()
which_last = target.index            # users, in target order

def val_mae(lam: float, k: int | None = None) -> float:
    g = hist_gaps.groupby("user")["gap"]
    if k is not None:
        g = hist_gaps.groupby("user").tail(k).groupby("user")["gap"]   # each user's most recent k gaps
    m_u = g.median().reindex(which_last)
    n_u = g.size().reindex(which_last)

    w = n_u / (n_u + lam)                                        # one weight per user, in [0, 1)
    log_pred = w * np.log(m_u) + (1 - w) * np.log(m_global)      # one blended log-prediction per user

    pred = np.exp(log_pred)
    return float(np.abs(target.to_numpy() - pred.to_numpy()).mean())

rows = []
for k in [None, 3, 5, 10]:
    for lam in [0, 1, 2, 5, 10, 20]:
        mae = val_mae(lam, k)
        rows.append({"k": k or "all", "lam": lam, "mae": mae, "vs_base_%": (1 - mae / 12_052_136) * 100})
pd.DataFrame(rows).sort_values("mae").head(10)

,k,lam,mae,vs_base_%
20,10,2,1.016555e+07,15.653517
19,10,1,1.017410e+07,15.582565
2,all,2,1.017904e+07,15.541598
13,5,1,1.018459e+07,15.495594
1,all,1,1.019062e+07,15.445516
14,5,2,1.019237e+07,15.430992
7,3,1,1.019933e+07,15.373241
8,3,2,1.025465e+07,14.914261
3,all,5,1.042269e+07,13.519996
21,10,5,1.042362e+07,13.512286


In [9]:
rows = []
for k in [10, 20, None]:
    for lam in [0.5, 1, 1.5, 2, 3]:
        mae = val_mae(lam, k)
        rows.append({"k": k or "all", "lam": lam, "mae": mae, "vs_base_%": (1 - mae / 12_052_136) * 100})
pd.DataFrame(rows).sort_values("mae").head(10)

,k,lam,mae,vs_base_%
2,10,1.5,1.013722e+07,15.888575
7,20,1.5,1.014591e+07,15.816523
12,all,1.5,1.015260e+07,15.760991
3,10,2.0,1.016555e+07,15.653517
8,20,2.0,1.017258e+07,15.595210
1,10,1.0,1.017410e+07,15.582565
13,all,2.0,1.017904e+07,15.541598
6,20,1.0,1.018380e+07,15.502108
11,all,1.0,1.019062e+07,15.445516
4,10,3.0,1.025731e+07,14.892168


In [10]:
gaps = s.assign(gap=s.groupby("user")["timestamp"].diff()).dropna(subset=["gap"])
m_global = float(gaps["gap"].median())          # from all gaps, as validated
if k is not None:
    gaps = gaps.groupby("user").tail(k)

In [11]:
# history = everything except each user's last row; held = that last row (the "next" interaction)
s = train.sort_values(["user", "timestamp"], kind="stable")
is_last = ~s["user"].duplicated(keep="last")
hist, held = s[~is_last], s[is_last].set_index("user")

t_last = hist.groupby("user")["timestamp"].max().reindex(held.index)
target = held["timestamp"] - t_last

pred = pd.Series(predict_gaps(fit_gap_model(hist, k=10, lam=1.5), held.index), index=held.index)
mae = lambda p: float((target - p).abs().mean())
print(f"current:            {mae(pred):,.0f}   (expect ~10.14M)")

# lower bound: the next book can't be read before anyone had it
first_seen = hist.groupby("item")["timestamp"].min()
lo = held["item"].map(first_seen) - t_last          # NaN where the book never appears in history
applies = lo > pred
pred_lo = pred.where(~applies, lo)
print(f"lower bound applies: {applies.mean():.1%} of users")
print(f"with lower bound:   {mae(pred_lo):,.0f}")

# upper bound: the gap can't run past the end of the data
t_end = clip_timestamps(s["timestamp"]).max()
hi = t_end - t_last
print(f"with both bounds:   {mae(pred_lo.clip(upper=hi)):,.0f}")

current:            10,137,223   (expect ~10.14M)
lower bound applies: 13.1% of users
with lower bound:   8,531,601
with both bounds:   10,071,869


In [12]:
first_seen_c = clip_timestamps(hist["timestamp"]).groupby(hist["item"]).min()
t_last_c = clip_timestamps(hist["timestamp"]).groupby(hist["user"]).max().reindex(held.index)
lo_c = held["item"].map(first_seen_c) - t_last_c
print(f"with clipped lower bound: {mae(pred.where(~(lo_c > pred), lo_c)):,.0f}")

with clipped lower bound: 8,541,255


In [13]:
bound = item_first_seen_bound(hist, held.index, held["item"])
print(mae(np.fmax(pred.to_numpy(), bound)))      # should print 8,531,601

8531600.701846221


In [14]:
extra = (target - lo)[applies]                      # how much later than the bound users actually read
c = extra.median()
print(f"median extra delay: {c / 86400:.1f} days")
for scale in [0, 0.5, 1.0, 1.5]:
    p = pred.where(~applies, lo + scale * c)
    print(f"bound + {scale:.1f}×median delay: {mae(p):,.0f}")

median extra delay: 35.0 days
bound + 0.0×median delay: 8,531,601
bound + 0.5×median delay: 8,474,746
bound + 1.0×median delay: 8,458,451
bound + 1.5×median delay: 8,469,957


In [15]:
print(mae(np.fmax(pred.to_numpy(), bound + 35 * 86400)))

8457130.18004211


In [16]:
OFFSET = 35 * 86400
bound = item_first_seen_bound(hist, held.index, held["item"])

rows = []
for k in [5, 10, 20, None]:
    for lam in [0.5, 1, 1.5, 2, 3, 5]:
        p = predict_gaps(fit_gap_model(hist, k=k, lam=lam), held.index)
        rows.append({"k": k or "all", "lam": lam, "mae": mae(np.fmax(p, bound + OFFSET))})
pd.DataFrame(rows).sort_values("mae").head(8)

,k,lam,mae
9,10,2.0,8.426265e+06
3,5,2.0,8.432098e+06
15,20,2.0,8.433124e+06
10,10,3.0,8.438330e+06
21,all,2.0,8.438338e+06
16,20,3.0,8.443347e+06
22,all,3.0,8.448219e+06
8,10,1.5,8.457130e+06
